# AlphaTrain Pillar 2f: Asymmetric Joint Training (Colab)

Shared backbone, but value gradients are **1000x downweighted** (val_weight=0.001)
to prevent the "backbone war" that killed iterations 1a-1c.

Policy head does 99.9% of backbone training. Value head learns ranking as a passive observer.

**Upload to Drive (`MyDrive/alphatrain/`):**
1. `colorlines_selfplay_train.tar.gz` — code (84 KB, same as before)
2. `alphatrain_pairwise.pt` — expert data (already on Drive)
3. `alphatrain_td_best.pt` — base model (already on Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

DRIVE = '/content/drive/MyDrive/alphatrain'

# Extract code
!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

# Copy data + model (both should already be on Drive from previous runs)
os.makedirs('/content/alphatrain/data', exist_ok=True)
for f in ['alphatrain_pairwise.pt', 'alphatrain_td_best.pt']:
    src = os.path.join(DRIVE, f)
    dst = os.path.join('/content/alphatrain/data', f)
    # Always re-copy from Drive (force fresh)
        print(f'Copying {f}...')
        shutil.copy(src, dst)
    print(f'{f}: {os.path.getsize(dst)/1e6:.0f} MB')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    if g.total_memory < 20e9:
        print('WARNING: T4 may not have enough VRAM. Use A100 runtime.')

!cd /content && python -m pytest alphatrain/tests/test_model.py alphatrain/tests/test_observation.py -v --tb=short 2>&1 | tail -10

In [ ]:
# Pillar 2f: Asymmetric Joint Training
# val_weight=0.001 — value gradients are 1000x downweighted
# Policy does 99.9% of backbone training, value learns as passive observer
# Pairwise ranking + anchor MSE for value head
# Warm start from epoch 6 model (policy=314 baseline)
%cd /content
!python -m alphatrain.train \
    --tensor-file alphatrain/data/alphatrain_pairwise.pt \
    --gpu-data --amp --compile \
    --value-bins 1 --val-weight 0.001 --anchor-weight 0.001 --rank-weight 1.0 \
    --resume alphatrain/data/alphatrain_td_best.pt --warm-start \
    --epochs 10 --batch-size 8192 --lr 1e-4 --warmup-epochs 2 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar2f_best.pt \
    --save-dir /content/checkpoints/pillar2f

In [ ]:
# Evaluate policy (baseline was 314 — should stay close or improve)
%cd /content
!python -m alphatrain.evaluate --player policy \
    --model /content/checkpoints/pillar2f/best.pt \
    --games 50 --seed 42

In [ ]:
# Copy final model to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/pillar2f/{f}'
    dst = f'{DRIVE}/pillar2f_{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {dst}')